## DeepISA tutorial

DeepISA is an end-to-end Python package designed to train DNA sequence model and quantify transcription factor (TF) cooperativity across the genome. It trains dilated convolutional neural networks on functional genomics data, such as CAGE-seq/ATAC-Seq/DNase-Seq/SuRE-Seq/STARR-Seq. The package then performs "In Silico Ablation" (ISA), a process that computationally masks specific TF binding motifs to measure how their individual and joint ablation affects predicted regulatory activity. By comparing these effects, deepISA categorizes TF interactions along a continuum from redundancy to synergy, revealing consistent biological rules of gene regulation.


To run the deepISA pipeline, you need to prepare the following files to run the deepISA pipeline:


1. Mandatory Genomic Resources

* Reference Genome (.fa): A FASTA file of the genome (the tutorial uses hg38). This is required to convert genomic locations to actual sequences

* TF Binding Annotations (.bb): Specifically, JASPAR 2026 genome-wide predictions in bigBed format. This allows the tool to know where the transcription factor motifs are located for ablation.


2. Primary Input Data: regulatory region and signal

You need your candidate regulatory regions and their activity levels. You have two main ways to provide this:

* Option A (Pre-quantified): A CSV or BED file containing genomic coordinates (chrom, start, end) and a column with pre-calculated signal intensities (e.g., TPM or read counts from CAGE-seq). Make sure that the signal intensity is NOT log-transformed.

* Option B (BigWig signal Tracks): A BED file of coordinates plus the corresponding BigWig (.bw) files. The pipeline will then quantify the signals for you.


3. Optional Enrichment Data

* ReMap 2022 (.bed): A curated atlas of ChIP-seq peaks. You use this if you want to filter your results to only include motifs that have ChIP-Seq evidence of binding in a specific cell type.

* Expressed TF List: A list of TFs known to be active in your cell line. This is highly recommended to reduce false positives by ensuring you only analyze TFs that are actually present.

4. Pre-trained Model (Alternative to Training):

* Weights File (.pt): If you don't want to train a new model from scratch, you can provide a pre-trained PyTorch model (like the model_blympho.pt mentioned) to go straight to the In Silico Ablation (ISA) analysis.

## Basic imports and data loading

Prepare the following files for the deepISA pipeline:

* Reference genome: [hg38](https://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz)

* Jaspar: [JASPAR 2026](https://mencius.uio.no/JASPAR/JASPAR_TFBSs/2026/JASPAR2026_hg38.bb) is genome-wide predictions of TF binding sites in bigBed format for hundreds of TFs, based on high-quality TF Position Frequency Matrices. Other versions (JASPAR 2022, 2024) may cause compatibility issues.

* ReMap (optional): [ReMap 2022](https://remap.univ-amu.fr/storage/remap2022/hg38/MACS2/remap2022_nr_macs2_hg38_v1_0.bed.gz) is a curated atlas of 182 million DNA-TF binding peaks, derived from 8103 human ChIP-seq datasets, covering hundreds of TFs across hundreds of cell lines and tissues.

In [1]:
import os
import pandas as pd
from loguru import logger

FASTA_PATH = "/maps/projects/ralab/people/pcr980/Resource/hg38.fa"
JASPAR_BB = "/maps/projects/ralab/people/pcr980/Resource/JASPAR/JASPAR2026_hg38.bb"
CELL_FACET = "B_lymphoblastoid"


## QuickStart, the fast track

regions_pos.bed comes containing randomly selected 1000 candidate regulatory regions with CAGE signal, for B_lymphoblastoid cell line. 

Subsetting regulatory regions aims to accelerate training for tutorial. The model performance is supposed to improve with full set of regulatory regions.

On output file names:
* "motif_": information about individual TF binding sites
* "tf_": TF features aggregated from all binding sites

In [2]:
# if you don't have a pretrained model:
from deepISA.utils import get_data_resource

bed_path = get_data_resource("regions_pos_with_count.csv")
df_regions = pd.read_csv(bed_path)
df_regions.head()

,chrom,start,end,target_reg
0,chr12,55725882,55726482,13.0
1,chr19,36028217,36028817,13.0
2,chr1,7403363,7403963,13.0
3,chr2,68062734,68063334,337.0
4,chr7,151632560,151633160,54.0


In [3]:
# if you don't have a pretrained model:

from deepISA.quickstart import QuickStart

pipe = QuickStart(
    results_dir="./QuickStart_results",
    fasta_path=FASTA_PATH,
    df_regions=df_regions,
    target_reg_col="target_reg", # specify the column in df_regions that contains the signal, the regression prediction target
)

pipe.train()


TypeError: __init__() got an unexpected keyword argument 'target_reg_col'

In [4]:
pipe.run_isa(
    jaspar_path=JASPAR_BB
)
pipe.aggregate_isa()
pipe.report() # Generate DeepCompAREs-stype plots demonstrating the cooperativity rules.

NameError: name 'pipe' is not defined

While `jaspar_path` is the only required argument for `.run_isa()`, providing a list of `expressed_tfs` is highly recommended for biological relevance. By passing a list of TFs known to be expressed in your specific cell line, the model filters out motifs of TFs that aren't present and therefore cannot be functional. We also provide function `get_expressed_tfs()` to estimate expressed TF list from the candidate regions and genome-wide bigwig signals. See section _____ for more information.

Variations of QuickStart usage:
* If you only have the regions and a bigwig file, but you haven't quantified it yourself, we can quantify it for you (also see section _____), just privide the bw_path in `.train()`.

* If you have a pretrained model already, just provide the model path, and we start from `.run_isa()`.

In [5]:
# If you have a pretrained model:
pretrained_model_path = get_data_resource("model_blympho.pt") # This is a model trained on the all the candidate CREs of B_lymphoblastoid, so we can skip training and go straight to ISA.
pipe = QuickStart(
    results_dir="./QuickStart_pretrained",
    model_path=pretrained_model_path, # path to the pretrained model
    fasta_path=FASTA_PATH,
    df_regions=df_regions,
)

# Skip .train()
pipe.run_isa(jaspar_path=JASPAR_BB)
pipe.aggregate_isa()
pipe.report()

22:17:48 | INFO     | Logger initialized. All logs and errors redirected to: ./QuickStart_pretrained/workflow.log


22:17:48 | INFO     | Using GPU 0: NVIDIA RTX 2000 Ada Generation Laptop GPU (7.65 GB)


22:17:48 | INFO     | Using device: cuda:0


22:17:48 | INFO     | Running ISA scenario: Threshold=500, RemapSubset=False


22:17:49 | INFO     | Starting motif mapping.


[urlOpen] Couldn't open /maps/projects/ralab/people/pcr980/Resource/JASPAR/JASPAR2026_hg38.bb for reading
[urlOpen] Couldn't open /maps/projects/ralab/people/pcr980/Resource/JASPAR/JASPAR2026_hg38.bb for reading
[pyBwOpen] bw is NULL!


RuntimeError: Received an error during file opening!

## Details of workflow: step by step

Let's disect the workflow that QuickStart wraps up.

In [6]:
# set up logger to save logs in the results directory
from deepISA.utils import setup_logger
from deepISA.modeling.preprocess import compile_training_data
from deepISA.utils import get_data_resource

RESULTS_DIR = "Step_by_step_res"
os.makedirs(RESULTS_DIR, exist_ok=True)

setup_logger(RESULTS_DIR)
logger.info("Starting deepISA tutorial")


22:17:49 | INFO     | Logger initialized. All logs and errors redirected to: Step_by_step_res/workflow.log


22:17:49 | INFO     | Starting deepISA tutorial


### 1. Prepare data for deep learning

This step uses function `compile_training_data()`. It takes in your BED file as a `pandas.DataFrame` and returns labeled data for downstream deep learning.

The input DataFrame must include the following column names:

* "chrom": chromosome identifier
* "start": start coordinate
* "end": end coordinate

Ensure that all intervals satisfy: end - start == target_input_length

Multiple preprocessing workflows are supported. You may either provide pre-quantified data (option 1), or use the built-in labeling function, simply by passing additional arguments to `compile_training_data()` (option 2).

#### Option 1: BED file with pre-quantified signals

Use this mode if your have already quantified the signals (e.g., TPM or read counts) for the candidate regions. The pipeline will automatically sample non-functional background regions to balance the dataset, so that number of positive and negative regions equalize.

These background regions are sampled genome-wide, excluding candidate Cis-Regulatory Elements ([cCREs](https://www.nature.com/articles/s41586-025-09909-9)), exons and the ENCODE blacklist.

Make sure that your quantified signals are not log-transformed, since compile_training_data() will automatically perform log transformation

In [7]:
# read an internal example file
df_pos = pd.read_csv(get_data_resource("regions_pos_with_count.csv"))
df_pos.head()

,chrom,start,end,target_reg
0,chr12,55725882,55726482,13.0
1,chr19,36028217,36028817,13.0
2,chr1,7403363,7403963,13.0
3,chr2,68062734,68063334,337.0
4,chr7,151632560,151633160,54.0


In [8]:
df_data = compile_training_data(
    df=df_pos, 
    seq_len=600, 
    target_reg_col="target_reg", # your column name for TPM/count/signal, not log transformed.
    outpath = os.path.join(RESULTS_DIR, "training_data.csv")
)

df_data.head() 

22:17:49 | INFO     | All 1000 regions already have the identical target length of 600 bp. Skipping resize.


22:17:49 | INFO     | Scenario 1: Using provided signal column 'target_reg'.


22:17:49 | INFO     | No class column provided. Inferring target_class (signal > 0).


22:17:49 | INFO     | Initial counts: 1000 positives, 0 negatives.


22:17:49 | INFO     | Sampling 1000 extra negatives from background pool.


22:17:49 | INFO     | Regions have variable lengths. Centering and resizing 1000 regions to 600 bp.


22:17:49 | INFO     | Saved training data to Step_by_step_res/training_data.csv


,chrom,start,end,target_reg,target_class
0,chr18,10716184,10716784,0.0,0.0
1,chr20,62937693,62938293,2329.0,1.0
2,chr7,145897560,145898160,0.0,0.0
3,chr19,11024900,11025500,15.0,1.0
4,chr4,140682613,140683213,0.0,0.0


Two additional columns are added by function `compile_training_data()`: 
* "target_reg": regression target, meaning the signal intensity before log transformation
* "target_class": classification target, 1 for positive regions (regulatory elements), 0 for negative regions.

#### Option 2: BED file + BigWig signal tracks

If you have genomic coordinates of regulatory elements, and want to quantify signals directly from BigWig files. The pipeline estimates a noise threshold from background regions to define the classification target.

Just pass a list of bigwig file paths to the optional parameter `bw_paths`.
  

#### Option 3: simply a bed file of positive regions
#TODO: to be implemented


### 2 Train convolutional neural network

This step uses function `train_model()`. It takes in the labeled data frame, performs train-val-test split, then train a exponentially dilated convolutional neural network. The main architecture highly resembles BPNet, but doesn't contain the per-base profile prediction for faster training.

Early stopping is enabled by default, so it is design to "overfit" the validation set. Use `evaluate_model()` to get the unbiased performance estimate.

By default, `rc_aug==True`, to apply reverse complement data augmentation. This is highly recommended for stable training, unless you are training on strand-aware data like single-stranded CAGE-Seq signals.

For hyperparameter tuning, `train_model()` will automatically save a "training_log.csv" to record the learning curve.

In [9]:

from deepISA.modeling.train import train_model

df_data = pd.read_csv(os.path.join(RESULTS_DIR, "training_data.csv"))

model, history, test_ds = train_model(
    df=df_data,
    fasta_path=FASTA_PATH, 
    model_dir=RESULTS_DIR,
    epochs=10    
)

# If you are satisfied with the data preprocessing but want to retrain the model
# just set use_cached_data=True

22:17:49 | INFO     | Using GPU 0: NVIDIA RTX 2000 Ada Generation Laptop GPU (7.65 GB)


22:17:49 | INFO     | Loading FASTA from /maps/projects/ralab/people/pcr980/Resource/hg38.fa


OSError: file `/maps/projects/ralab/people/pcr980/Resource/hg38.fa` not found

In [10]:
# To get real test performance, run:
from deepISA.modeling.predict import evaluate_model
test_r = evaluate_model(model, test_ds)
print(f"Final Test Pearson r: {test_r}")

NameError: name 'model' is not defined

### 3. Map motifs for all candidate regulatory regions with positive signal

This step uses function `map_motif()`. It takes in the candidate regulatory regions with positive signal, and returns all the motif locations according to [JASPAR 2026](https://mencius.uio.no/JASPAR/JASPAR_TFBSs/2026/JASPAR2026_hg38.bb) database, thresholded by motif matching score, for each regulatory region. 

The JASPAR matching score (range: 100-1000) quantifies how well a short DNA sequence aligns with a transcription factor motif from the JASPAR database, using a position weight matrix to compute a similarity score at each position. The default Jaspar score is 500, meaning the p-value of motif matching is 1e-5.

In [11]:
from deepISA.scoring.annotation import map_motifs

#TODO: write preprocess_remap(cell_type). 
motif_csv_path = map_motifs(
    regions_df=df_pos, 
    jaspar_path=JASPAR_BB,
    expressed_tfs=None, # set to None for tutorial, but providing expressed_tf_list is highly recommended for real analysis to reduce false positives.
    outpath=os.path.join(RESULTS_DIR,"motif_locs.csv"),
    score_thresh=500,
    remap_path=None # don't specify whether a TFBS is supported by ChIP-Seq due to lack of cell-type specific ChIP-Seq data in remap 
)

# read the mapped motif locations
df_motif_locs = pd.read_csv(os.path.join(RESULTS_DIR,"motif_locs.csv"))
df_motif_locs.head()

22:17:49 | INFO     | Starting motif mapping.


[urlOpen] Couldn't open /maps/projects/ralab/people/pcr980/Resource/JASPAR/JASPAR2026_hg38.bb for reading
[urlOpen] Couldn't open /maps/projects/ralab/people/pcr980/Resource/JASPAR/JASPAR2026_hg38.bb for reading
[pyBwOpen] bw is NULL!


RuntimeError: Received an error during file opening!

## 3.5 (Optional) Filter spurious motifs with attribution-based filtering

After `map_motifs()`, the mapped motif locations may contain false positives — motifs that match the JASPAR position weight matrix but are not actually bound by the TF in vivo. We provide an optional **non-motif null attribution filter** that uses DeepLIFT attribution on the trained model to distinguish true motif instances from spurious ones.

**How it works:**
1. For each region, generate windows covering non-motif sequence (null regions)
2. Compute DeepLIFT attribution on these null windows to build a background distribution
3. For each candidate motif, compute its attribution score
4. Motifs whose attribution falls below a computed threshold (based on the null distribution) are filtered out as spurious

**When to use:**
- When you have a well-trained model with good predictive performance
- When you observe many low-confidence motifs that may be noise
- Before running ISA to reduce false positives in cooperativity analysis

**Parameters:**
- `window_size=20`: Size of non-motif windows for null distribution
- `stride=20`: Stride for sliding windows
- `device='cuda'`: Compute device (use 'cpu' if no GPU available)

In [12]:
import torch
from deepISA.filters.pipeline import run_pipeline as run_filter_pipeline
from deepISA.filters.utils.io import save_filtered_motifs
from deepISA.utils import find_available_gpu, prepare_chroms_from_fasta
import os

# ============================================================
# FILTER PARAMETERS (adjust as needed)
# ============================================================
FILTER_WINDOW_SIZE = 20    # Non-motif window size (bp)
FILTER_STRIDE = 20        # Window stride (bp)
FILTER_N_REGIONS = None   # None = all regions, or set integer to limit

# Auto-detect genome: use existing chr*.fa or generate from FASTA_PATH
# deepISA uses FASTA_PATH (single hg38.fa), filter needs chr*.fa files
_filter_genome_dir = "./data/genome/hg38/chroms"
if not os.path.exists(os.path.join(_filter_genome_dir, "chr1.fa")):
    print(f"[filter] Generating chr*.fa files from {FASTA_PATH} ...")
    _filter_genome_dir = prepare_chroms_from_fasta(
        fasta_path=FASTA_PATH,
        chroms_dir=_filter_genome_dir,
    )
    print(f"[filter] Genome prepared at: {_filter_genome_dir}")

FILTER_GENOME_DIR = _filter_genome_dir

# Use GPU if available, otherwise CPU
device = find_available_gpu()
print(f"Using device: {device}")

# Run the attribution-based motif filter
# This uses the same model trained above to filter out spurious motifs
df_motif_locs_filtered = run_filter_pipeline(
    model=model,
    regions_df=df_pos,
    motif_locs_df=df_motif_locs,
    fasta_dir=FILTER_GENOME_DIR,  # Directory containing chr*.fa files
    seq_len=600,
    window_size=FILTER_WINDOW_SIZE,
    stride=FILTER_STRIDE,
    device=device,
    n_regions=FILTER_N_REGIONS,
)

# Save filtered motifs for downstream ISA analysis
filtered_motif_path = os.path.join(RESULTS_DIR, "motif_locs_filtered.csv")
save_filtered_motifs(df_motif_locs_filtered, filtered_motif_path)

print(f"\n=== Motif Filter Results ===")
print(f"Input motifs:  {len(df_motif_locs)}")
print(f"Passed motifs: {len(df_motif_locs_filtered)}")
print(f"Pass rate:     {len(df_motif_locs_filtered)/len(df_motif_locs)*100:.1f}%")
print(f"Filtered motifs saved to: {filtered_motif_path}")


22:17:49 | INFO     | Using GPU 0: NVIDIA RTX 2000 Ada Generation Laptop GPU (7.65 GB)


Using device: cuda:0


NameError: name 'model' is not defined


One way to reduce false positive of TF binding site is to retain only those with overlapping ChIP-Seq peak recorded in [ReMap 2022](https://remap2022.univ-amu.fr/) database. It is essentially a giant bed file containing genomic locations of ChIP-Seq peak for thousands of TFs in hundreds of cell lines. By setting `remap_path=YOUR_REMAP_PATH`, function `map_motif()` will return an additional column "remap_evidence" of boolean values, indicating whether each motif is supported by the ChIP-Seq peak of the same TF according to ReMap. Then you can subset manually to contain only the motifs supported by ChIP-Seq.

However, not all TFs expressed in each cell line have a corresponding ChIP-Seq experiment, thus subsetting by ReMap evidence risks losing valid TF binding sites simply due to lack of ChIP-Seq experiment recorded in ReMap.

We recommend a more lenient motif filtering strategy: filtering by TF gene expression level since only expressed TFs can exert influence. By passing `expressed_tfs=YOUR_EXPRESSED_TF_LIST`, function `map_motif()` will only return the motifs of TFs in the `expressed_tfs`.

In [13]:
from deepISA.scoring.infer_tf_expr import get_expressed_tfs

# Define the same BigWig tracks used for training
bw_paths = [
    f"{BW_DIR}/{CELL_FACET}.minus.bw", 
    f"{BW_DIR}/{CELL_FACET}.plus.bw"
]

# Infer the list of expressed TFs based on a noise threshold
expressed_tf_list = get_expressed_tfs(
    bw_paths=bw_paths, 
    percentile=99
)

len(expressed_tf_list)

NameError: name 'BW_DIR' is not defined

### 4. Run single ISA to quantify TF importance

This step first uses function `run_single_isa()`. It takes in a well-trained model and the filtered motif locations, and output the in silico ablation scores of all motifs directly to hard disk. This step is computation heavy, thus GPU is necessary here. 

Then we use function `calc_tf_importance()` to aggregate all the motif ISA scores for each TF. Aggregation methods include mean, median and signed Kolmogorov-Smirnov (KS) statistics. KS requires sufficient instances for accurate calculation, thus it produces a warning if fewer than 5 instances exist for a TF.

In [14]:
import torch
from deepISA.modeling.cnn import Conv
from deepISA.scoring.single_isa import run_single_isa, calc_tf_importance
from deepISA.utils import find_available_gpu

# 1. Load model, send to GPU if available
model=Conv()
model_path = f"{RESULTS_DIR}/model_best.pt"
model.load_state_dict(torch.load(model_path, weights_only=False))
device = find_available_gpu()

# 2. read the mapped motif locations
motif_locs=pd.read_csv(os.path.join(RESULTS_DIR, "motif_locs.csv"))

# 3. Compute Single-Motif ISA
run_single_isa(
    model=model, 
    fasta_path=FASTA_PATH,
    motif_locs=motif_locs,
    device=device,
    outpath=os.path.join(RESULTS_DIR, "motif_single_isa.csv"),
    batch_size=200  # Adjust based on your GPU memory, larger batch size will speed up the process but use more memory
)

FileNotFoundError: [Errno 2] No such file or directory: 'Step_by_step_res/model_best.pt'

In [15]:
df_single_isa = pd.read_csv(os.path.join(RESULTS_DIR, "motif_single_isa.csv"))
df_single_isa.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Step_by_step_res/motif_single_isa.csv'

In [16]:
# 4: aggregate single ISA
tf_importance = calc_tf_importance(os.path.join(RESULTS_DIR, "motif_single_isa.csv"))

# Save the final summary
tf_importance.to_csv(os.path.join(RESULTS_DIR, "tf_importance.csv"), index=False)

# 5. Visualize Top Drivers (Track 0: Regression/Expression)
print("Top 10 TFs by Expression Impact:")
display(tf_importance.sort_values("ks_isa_t0", ascending=False).head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'Step_by_step_res/motif_single_isa.csv'

### 5. Run combinatorial ISA for TF pair cooperativity

This step first uses function `run_combi_isa()`. It takes in a well-trained model, the filtered motif locations, and model receptive field. Then it performs combinatorial ISA for all motifs on same regulatory region within distance of the model receptive field. Models trained by deepISA package have attribute `.rf` for receptive field.

Then we use function `calc_coop_score()` to aggregate the raw combinatorial ISA results and obtain cooperativity score. Note, this aggregation can be performed on both TF-pair and TF level, by specifying `level=tf_pair`, or `level='tf_pair`. 

In the cooperativity score report, TF pairs are always alphabetically sorted and concatenated by "|".

In [17]:
from deepISA.scoring.combi_isa import run_combi_isa

# 1. Run the pairwise ablation
COMB_RESULTS_PATH = os.path.join(RESULTS_DIR, "motif_combi_isa.csv")

run_combi_isa(
    model=model,
    fasta_path=FASTA_PATH,
    motif_locs=motif_locs, 
    device=device,
    receptive_field=model.rf,
    outpath=COMB_RESULTS_PATH
)

df_combi_isa = pd.read_csv(COMB_RESULTS_PATH)
df_combi_isa.head()


NameError: name 'motif_locs' is not defined

In [18]:
from deepISA.scoring.combi_isa import  calc_coop_score

outpath_pair = os.path.join(RESULTS_DIR, "coop_tf_pair.csv")
calc_coop_score(df_combi_isa, outpath_pair, level="tf_pair")

outpath_tf = os.path.join(RESULTS_DIR, "coop_tf.csv")
calc_coop_score(df_combi_isa, outpath_tf, level="tf")


NameError: name 'df_combi_isa' is not defined

### 6. Plot results

deepISA package provide deepCompARE style analysis and plots, for reproducibility and sanity check.

In [19]:
# load result data
import os
import pandas as pd
RESULTS_DIR = "Step_by_step_res"
df_raw_combi = pd.read_csv(os.path.join(RESULTS_DIR, "motif_combi_isa.csv"))
df_coop_pair = pd.read_csv(os.path.join(RESULTS_DIR, "coop_tf_pair.csv"))
df_coop_tf = pd.read_csv(os.path.join(RESULTS_DIR, "coop_tf.csv"))


FileNotFoundError: [Errno 2] No such file or directory: 'Step_by_step_res/motif_combi_isa.csv'

#### Observation 1: interaction decays along distance

In [20]:
from deepISA.plotting.interaction import (
    plot_null,
    plot_interaction_decay, 
)

plot_null(df_raw_combi)

NameError: name 'df_raw_combi' is not defined

In [21]:
plot_interaction_decay(df_raw_combi)

NameError: name 'df_raw_combi' is not defined

#### Observation 2: TF-pair level cooperativity 

In [22]:
from deepISA.plotting.cooperativity import (
    hist_coop_score,
    heatmap_coop_score,
    plot_motif_distance_by_category
)

hist_coop_score(df_coop_pair)

NameError: name 'df_coop_pair' is not defined

In [23]:
# Height for labels can be set based on max frequency
heatmap_coop_score(df_coop_pair)

NameError: name 'df_coop_pair' is not defined

In [24]:
plot_motif_distance_by_category(df_coop_pair)

NameError: name 'df_coop_pair' is not defined

#### Observation 3: TF level cooperativity

In [25]:
from deepISA.plotting.tf import (
    plot_motif_gc_by_coop,
    plot_coop_vs_importance
)

df_coop_tf = pd.read_csv(os.path.join(RESULTS_DIR, "coop_tf.csv"))
plot_motif_gc_by_coop(df_coop_tf)

FileNotFoundError: [Errno 2] No such file or directory: 'Step_by_step_res/coop_tf.csv'

In [26]:
# Plot cooperativity score vs importance score 
df_importance = pd.read_csv(os.path.join(RESULTS_DIR, "tf_importance.csv"))
plot_coop_vs_importance(df_coop_tf, df_importance)

FileNotFoundError: [Errno 2] No such file or directory: 'Step_by_step_res/tf_importance.csv'

### 7. Explore association with external data

Tell user dimers "::" are removed from PPI validation.
Only for human

#### Observation 4: TF family behavior

In [27]:
from deepISA.validating.tf_family import (
    plot_coop_by_tf_pair_family,
    plot_coop_by_dbd,
    plot_intra_family_coop_score
)

df_res=plot_coop_by_tf_pair_family(df_coop_pair)

NameError: name 'df_coop_pair' is not defined

In [28]:
df_res=plot_intra_family_coop_score(df_coop_pair)

NameError: name 'df_coop_pair' is not defined

In [29]:
df_res=plot_coop_by_dbd(df_coop_tf)

NameError: name 'df_coop_tf' is not defined

In [30]:
from deepISA.validating.tf_pair_ppi import (
    plot_ppi_enrichment,
    validate_cofactor_recruitment
)


df_res = plot_ppi_enrichment(df_coop_pair)

NameError: name 'df_coop_pair' is not defined

In [31]:
# Cooperativity between TF pairs of same family
df_res=validate_cofactor_recruitment(df_coop_pair)

NameError: name 'df_coop_pair' is not defined

Validate with TF functions: gini index and known TF functional categories.

In [32]:
from deepISA.validating.tf_function import (
    plot_usf_pfs,
    plot_cell_specificity
)

plot_usf_pfs(df_coop_tf)

NameError: name 'df_coop_tf' is not defined

In [33]:
plot_cell_specificity(df_coop_tf)

NameError: name 'df_coop_tf' is not defined